## 1. Setup

In [1]:
!pip install -q open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.7 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import torch
import open_clip
from pathlib import Path

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

SEED = 42

MODEL_NAME = "hf-hub:Marqo/marqo-fashionCLIP"

model, _, _ = open_clip.create_model_and_transforms(
    MODEL_NAME
)

tokenizer = open_clip.get_tokenizer(
    MODEL_NAME
)

model = model.to(DEVICE)
model.eval()

print("Device:", DEVICE)
print("Model:", MODEL_NAME)
print("Embedding dimension:", model.visual.output_dim)
print("Context length:", model.context_length)

open_clip_config.json:   0%|          | 0.00/532 [00:00<?, ?B/s]

open_clip_model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Device: cuda
Model: hf-hub:Marqo/marqo-fashionCLIP
Embedding dimension: 512
Context length: 77


## 2. Load Cached Image Embeddings

In [3]:
IMAGE_EMBEDDINGS_PATH = "/kaggle/input/notebooks/jndhruv/03a-image-embeddings/image_embeddings.npy"
IMAGE_EMBEDDING_IDS_PATH = "/kaggle/input/notebooks/jndhruv/03a-image-embeddings/image_embedding_ids.npy"

image_embeddings = torch.from_numpy(
    np.load(IMAGE_EMBEDDINGS_PATH)
)

image_embedding_ids = np.load(
    IMAGE_EMBEDDING_IDS_PATH
)

print("Image embeddings:", image_embeddings.shape)
print("Image IDs:", image_embedding_ids.shape)

Image embeddings: torch.Size([44419, 512])
Image IDs: (44419,)


In [4]:
assert image_embeddings.ndim == 2
assert image_embeddings.shape[1] == 512
assert len(image_embeddings) == len(image_embedding_ids)

print("Image embedding artifact validated.")

Image embedding artifact validated.


## 3. Load Canonical Product Data

In [5]:
PRODUCTS_PATH = Path(
    "/kaggle/input/notebooks/jndhruv/02-data-cleaning/multi-modal-fashion-ecom/"
    "data/processed/products.parquet"
)

products_df = pd.read_parquet(PRODUCTS_PATH)

print("Products loaded:", len(products_df))

Products loaded: 44419


In [6]:
assert len(products_df) == 44_419

### Align product metadata to the exact image embedding order

In [7]:
embedding_lookup = products_df.set_index("id")

embedding_df = (
    embedding_lookup
    .loc[image_embedding_ids]
    .reset_index()
)

print("Embedding dataframe:", embedding_df.shape)

Embedding dataframe: (44419, 23)


In [8]:
assert len(embedding_df) == len(image_embeddings)

assert (
    embedding_df["id"].to_numpy()
    == image_embedding_ids
).all()

print("Product <--> image embedding alignment verified.")

Product <--> image embedding alignment verified.


## 4. Build Clip Text

In [9]:
def build_clip_text(row):
    parts = [f"{row['product_display_name']}."]
    
    fields = {
        "Brand": row["brand_name"],
        "Category": row["article_type"],
        "Colour": row["base_colour"],
        "Usage": row["usage"],
    }

    for label, value in fields.items():
        if pd.notna(value) and str(value).strip():
            parts.append(
                f"{label}: {str(value).strip()}."
            )

    return " ".join(parts)


embedding_df["clip_text"] = embedding_df.apply(
    build_clip_text,
    axis=1
)

In [10]:
clip_tokens = tokenizer(
    embedding_df["clip_text"].tolist()
)

clip_token_counts = (
    (clip_tokens != 0).sum(dim=1)
)

print("Token shape:", clip_tokens.shape)
print("Min:", clip_token_counts.min().item())
print("Max:", clip_token_counts.max().item())
print(
    "Mean:",
    clip_token_counts.float().mean().item()
)

at_limit = (
    clip_token_counts >= model.context_length - 2
).sum().item()

print("At/near limit:", at_limit)
print(
    "Percentage:",
    f"{100 * at_limit / len(embedding_df):.2f}%"
)

Token shape: torch.Size([44419, 77])
Min: 21
Max: 49
Mean: 27.754541397094727
At/near limit: 0
Percentage: 0.00%


### Recreate the evaluation sample

In [11]:
experiment_df = (
    embedding_df
    .groupby("article_type", group_keys=False)
    .apply(
        lambda x: x.sample(
            n=min(len(x), 5),
            random_state=SEED
        )
    )
    .reset_index(drop=True)
)

print("Experiment products:", len(experiment_df))

Experiment products: 661


/tmp/ipykernel_24/1494156851.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [12]:
embedding_index = pd.Series(
    np.arange(len(image_embedding_ids)),
    index=image_embedding_ids
)

experiment_indices = embedding_index[
    experiment_df["id"]
].to_numpy()

experiment_image_embeddings = (
    image_embeddings[experiment_indices]
)

print(
    "Experiment image embeddings:",
    experiment_image_embeddings.shape
)

Experiment image embeddings: torch.Size([661, 512])


In [13]:
assert (
    experiment_df["id"].to_numpy()
    == image_embedding_ids[experiment_indices]
).all()

## 5. Text Encoder

### Helper

In [14]:
def encode_texts(texts, batch_size=32):
    embeddings = []

    with torch.no_grad():
        for i in range(0, len(texts), batch_size):

            tokens = tokenizer(
                texts[i:i + batch_size]
            ).to(DEVICE)

            batch = model.encode_text(tokens)

            batch = (
                batch
                / batch.norm(
                    dim=-1,
                    keepdim=True
                )
            )

            embeddings.append(
                batch.cpu()
            )

    return torch.cat(embeddings, dim=0)

In [15]:
novel_queries = [
    "black running shoes",
    "white casual women's t-shirt",
    "ethnic women's summer outfit",
    "brown formal men's shoes",
    "floral printed cotton dress",
    "embroidered floral top",
]

query_embeddings = encode_texts(novel_queries)
print("Query embeddings:", query_embeddings.shape)

Query embeddings: torch.Size([6, 512])


In [16]:
def retrieve(query_emb, image_emb, top_k=5):
    sims = query_emb @ image_emb.T
    scores, indices = torch.topk(sims, k=top_k, dim=1)
    return scores, indices

scores, indices = retrieve(query_embeddings, experiment_image_embeddings)

for qi, query in enumerate(novel_queries):
    print(f"\n{'='*80}\nQUERY: {query}")
    for rank, idx in enumerate(indices[qi], start=1):
        row = experiment_df.iloc[idx.item()]
        print(f"{rank}. {row['product_display_name']} | {row['article_type']} | {row['base_colour']} | score={scores[qi, rank-1]:.4f}")


QUERY: black running shoes
1. Fila Men Trackfield Black Sports Shoes | Sports Shoes | Black | score=0.2657
2. Newfeel Unisex Black Casual Shoes | Casual Shoes | Black | score=0.2401
3. Red Tape Men's Casual Black Shoe | Casual Shoes | Black | score=0.2209
4. Nike Women Black Tenkay Shoes | Flats | Black | score=0.2157
5. ADIDAS Women Oregon 11 Navy Blue Sports Shoes | Sports Shoes | Navy Blue | score=0.1971

QUERY: white casual women's t-shirt
1. Mumbai Slang Women Printed Grey Top | Tops | Grey | score=0.1795
2. Palm Tree Girl's Linnea White Green Kidswear | Tops | Green | score=0.1652
3. Jockey White Camisole | Camisoles | White | score=0.1585
4. Fabindia Women White Salwar | Salwar | White | score=0.1499
5. W Women Beige Kurta | Kurtas | Beige | score=0.1483

QUERY: ethnic women's summer outfit
1. Biba Outlet Women White Printed Churidar Kurta with Dupatta | Kurta Sets | White | score=0.2458
2. Fabindia Girls Printed Blue Lehenga Choli | Lehenga Choli | Blue | score=0.2325
3. Vishu

## Conclusions

### 1. Text representation

The original `search_text` representation contains rich product information,
but FashionCLIP's text encoder has a 77-token context window. Across the
44,419-product catalogue, 32,896 (~74%) of `search_text` values reached the
context limit and were therefore truncated.

A compact `clip_text` representation was constructed using:

- product display name
- brand
- article type
- colour
- usage

This representation produced:

- minimum: 21 tokens
- maximum: 49 tokens
- mean: ~27.75 tokens
- 0% at/near the context limit

Therefore, `clip_text` allows the complete representation to reach the
FashionCLIP text encoder without truncation.

### 2. Initial self-retrieval experiment

An initial Recall@K experiment compared `search_text` and `clip_text`
by using each product's own text to retrieve its own image.

Although `search_text` achieved higher Recall@K, this evaluation was rejected
as the primary representation-selection criterion because it measured
product self-identification rather than generalization to novel user queries.
The richer `search_text` also contains more product-specific information,
creating a uniqueness advantage.

### 3. Novel-query sanity check

Novel fashion queries were evaluated against the image embedding sample.
The model demonstrated sensible semantic retrieval for broad queries such as
black running shoes, brown formal men's shoes, and ethnic women's clothing,
while more fine-grained queries such as embroidered floral top were less
reliable.

### 4. Decision

`clip_text` is selected as the compact catalogue-side representation for
FashionCLIP-related experimentation.

`search_text` remains the richer textual representation for the sparse/BM25
retrieval component.

The final dense retrieval implementation and fusion strategy are handled
separately by the retrieval pipeline.